<a href="https://colab.research.google.com/github/Maee127/Adversarial-Notebooks/blob/master/%238/Note08_Attention_Awareness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================
# Adversarial Attacks Series — Note 08
# Does the Model Know Where It Looks?
# Attention as a Test of Awareness
# =============================================================
#
# Series:  Humble Model / Attention & Interpretability
# Dataset: CIFAR-10
# Model:   Small Vision Transformer (trained from scratch)
#
# Notebook structure:
#   Part A: Imports and Setup
#   Part B: Dataset Loading (CIFAR-10, held-out split)
#   Part C: Small ViT Architecture
#   Part D: Training the ViT
#   Part E: Attention Extraction and Entropy Computation
#   Part F: Adversarial Attack (FGSM and PGD)
#   Part G: Evaluation with Deferral
#   Part H: Head-to-Head Comparison with Essays #6 and #7
#   Part I: Summary and Outputs
# =============================================================

In [1]:
# -------------------------------------------------------------
# Part A: Imports and Setup
# -------------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import random
import json
import math

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


Using device: cuda


In [2]:
# -------------------------------------------------------------
# Part B: Dataset Loading (CIFAR-10) — with held-out split
# -------------------------------------------------------------

cifar_transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

cifar_transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform_test)

calib_size = int(0.2 * len(test_dataset))
eval_size = len(test_dataset) - calib_size
calib_dataset, eval_dataset = random_split(test_dataset, [calib_size, eval_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset):,}")
print(f"Calibration samples: {len(calib_dataset):,}")
print(f"Evaluation samples: {len(eval_dataset):,}")

# Per-channel valid normalized range
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
CIFAR_MIN = ((0 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)
CIFAR_MAX = ((1 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)

def clamp_valid(x):
    return torch.max(torch.min(x, CIFAR_MAX), CIFAR_MIN)


100%|██████████| 170M/170M [52:17<00:00, 54.4kB/s]


Training samples: 50,000
Calibration samples: 2,000
Evaluation samples: 8,000


In [3]:
# -------------------------------------------------------------
# Part C: Small ViT Architecture
# -------------------------------------------------------------
#
# We patchify the 32x32 image into 4x4 patches -> 8x8 = 64 patches.
# Each patch is flattened to 3*4*4 = 48 dims, embedded to dim=128.
# A [CLS] token is prepended. Positional embeddings are learned.
# We use a custom Transformer block so attention weights are
# directly returnable, instead of relying on nn.TransformerEncoder.
# -------------------------------------------------------------

class ViTBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, return_attention=False):
        # Pre-norm attention
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.attn(x_norm, x_norm, x_norm, need_weights=True)
        x = x + attn_out
        # Pre-norm MLP
        x = x + self.mlp(self.norm2(x))
        if return_attention:
            return x, attn_weights
        return x, None

class SmallViT(nn.Module):
    def __init__(self, image_size=32, patch_size=4, num_classes=10,
                 dim=128, depth=4, heads=4, mlp_dim=256, dropout=0.1):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim = 3 * patch_size * patch_size

        self.patch_embed = nn.Linear(patch_dim, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, dim) * 0.02)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            ViTBlock(dim, heads, mlp_dim, dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def patchify(self, x):
        # x: [B, 3, 32, 32]
        B, C, H, W = x.shape
        p = self.patch_size
        # Unfold into patches
        patches = x.unfold(2, p, p).unfold(3, p, p)  # [B, C, H/p, W/p, p, p]
        patches = patches.contiguous().view(B, C, -1, p * p)  # [B, C, N, p*p]
        patches = patches.permute(0, 2, 1, 3).contiguous()  # [B, N, C, p*p]
        patches = patches.view(B, -1, C * p * p)  # [B, N, C*p*p]
        return patches

    def forward(self, x, return_attention=False):
        B = x.size(0)
        patches = self.patchify(x)                      # [B, N, patch_dim]
        tokens = self.patch_embed(patches)              # [B, N, dim]
        cls = self.cls_token.expand(B, -1, -1)          # [B, 1, dim]
        tokens = torch.cat([cls, tokens], dim=1)        # [B, N+1, dim]
        tokens = tokens + self.pos_embed
        tokens = self.dropout(tokens)

        attentions = []
        for block in self.blocks:
            tokens, attn = block(tokens, return_attention=return_attention)
            if return_attention:
                attentions.append(attn)  # [B, N+1, N+1]

        tokens = self.norm(tokens)
        logits = self.head(tokens[:, 0])                # CLS token
        if return_attention:
            return logits, attentions
        return logits

# Quick sanity check
_vit = SmallViT().to(DEVICE)
_x = torch.randn(2, 3, 32, 32).to(DEVICE)
_logits, _atts = _vit(_x, return_attention=True)
print(f"Logits shape: {_logits.shape}")
print(f"Number of attention layers returned: {len(_atts)}")
print(f"Attention shape (layer 0): {_atts[0].shape}")

Logits shape: torch.Size([2, 10])
Number of attention layers returned: 4
Attention shape (layer 0): torch.Size([2, 65, 65])


In [4]:
# -------------------------------------------------------------
# Part D: Training the ViT
# -------------------------------------------------------------

def train_vit(model, train_loader, epochs=30, lr=3e-4, weight_decay=0.05):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch} — avg loss: {avg_loss:.4f} — lr: {scheduler.get_last_lr()[0]:.6f}")
    return model

vit_model = SmallViT().to(DEVICE)

if os.path.exists('checkpoint_vit_cifar10.pth'):
    checkpoint = torch.load('checkpoint_vit_cifar10.pth', map_location=DEVICE)
    vit_model.load_state_dict(checkpoint['model_state_dict'])
    print("Loaded ViT model from checkpoint")
else:
    print("Training ViT from scratch...")
    vit_model = train_vit(vit_model, train_loader, epochs=30)
    torch.save({'model_state_dict': vit_model.state_dict()}, 'checkpoint_vit_cifar10.pth')
    print("Saved ViT model checkpoint")

def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

clean_acc = evaluate_accuracy(vit_model, eval_loader)
print(f"ViT clean test accuracy: {clean_acc:.2f}%")


Training ViT from scratch...


Epoch 0: 100%|██████████| 782/782 [00:32<00:00, 24.05it/s]


Epoch 0 — avg loss: 1.7308 — lr: 0.000299


Epoch 1: 100%|██████████| 782/782 [00:31<00:00, 24.45it/s]


Epoch 1 — avg loss: 1.4516 — lr: 0.000297


...

Epoch 27 — avg loss: 0.6561 — lr: 0.000003


Epoch 28: 100%|██████████| 782/782 [00:33<00:00, 23.01it/s]


Epoch 28 — avg loss: 0.6550 — lr: 0.000001


Epoch 29: 100%|██████████| 782/782 [00:33<00:00, 23.68it/s]


Epoch 29 — avg loss: 0.6489 — lr: 0.000000
Saved ViT model checkpoint
ViT clean test accuracy: 76.85%


In [5]:
# -------------------------------------------------------------
# Part E: Attention Extraction and Entropy Computation
# -------------------------------------------------------------
#
# For each sample, we extract the attention matrix from the
# [CLS] token (index 0) across all heads in a chosen layer.
# We average attention across heads, normalize, and compute
# entropy over patches (excluding the CLS token itself).
#
# High entropy  -> attention spread across many patches
# Low entropy   -> attention focused on a few patches
# -------------------------------------------------------------

def compute_attention_entropy(attentions, layer=-1, include_cls=False):
    """
    attentions: list of tensors [B, N+1, N+1] from SmallViT
    layer: which layer to use (-1 = last)
    include_cls: whether to include the CLS token in the entropy
    Returns: tensor [B]
    """
    attn = attentions[layer]                    # [B, N+1, N+1]
    cls_attention = attn[:, 0, :]               # [B, N+1] (CLS row)
    if not include_cls:
        cls_attention = cls_attention[:, 1:]    # drop CLS column
    # Normalize (row should already sum to 1, but be safe)
    cls_attention = cls_attention / (cls_attention.sum(dim=-1, keepdim=True) + 1e-10)
    entropy = -(cls_attention * (cls_attention + 1e-10).log()).sum(dim=-1)
    return entropy

def compute_attention_entropy_multihead(attentions, layers=(-1,)):
    """
    Average entropy across multiple layers (if requested).
    """
    entropies = [compute_attention_entropy(attentions, layer=l) for l in layers]
    return torch.stack(entropies, dim=0).mean(dim=0)

# Sanity check
vit_model.eval()
test_images, test_labels = next(iter(eval_loader))
test_images = test_images[:4].to(DEVICE)
with torch.no_grad():
    logits, atts = vit_model(test_images, return_attention=True)
entropies = compute_attention_entropy(atts)
print(f"Sanity check — attention entropy per sample: {entropies.cpu().numpy()}")


Sanity check — attention entropy per sample: [3.8247664 3.9046507 4.0216484 4.064683 ]


In [6]:
# -------------------------------------------------------------
# Part F: Adversarial Attack (FGSM and PGD)
# -------------------------------------------------------------

def fgsm_attack(model, images, labels, epsilon=0.03):
    model.eval()
    images_adv = images.clone().detach().to(DEVICE)
    images_adv.requires_grad = True
    logits = model(images_adv)
    loss = nn.CrossEntropyLoss()(logits, labels.to(DEVICE))
    model.zero_grad()
    loss.backward()
    perturbed = images_adv + epsilon * images_adv.grad.sign()
    return clamp_valid(perturbed).detach()

def pgd_attack(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    original = images.clone().detach().to(DEVICE)
    images_adv = original.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad = True
        logits = model(images_adv)
        loss = nn.CrossEntropyLoss()(logits, labels.to(DEVICE))
        model.zero_grad()
        loss.backward()
        grad_sign = images_adv.grad.sign()
        images_adv = images_adv + step_size * grad_sign
        perturbation = torch.clamp(images_adv - original, -epsilon, epsilon)
        images_adv = clamp_valid(original + perturbation).detach()
    return images_adv

# Quick check: clean vs. PGD accuracy on a sample
vit_model.eval()
sample_images, sample_labels = next(iter(eval_loader))
sample_images = sample_images.to(DEVICE)
sample_labels = sample_labels.to(DEVICE)
with torch.no_grad():
    clean_preds = vit_model(sample_images).argmax(dim=1)
    clean_correct = (clean_preds == sample_labels).float().mean().item()

pgd_images = pgd_attack(vit_model, sample_images, sample_labels)
with torch.no_grad():
    pgd_preds = vit_model(pgd_images).argmax(dim=1)
    pgd_correct = (pgd_preds == sample_labels).float().mean().item()

print(f"Sample clean accuracy: {clean_correct*100:.2f}%")
print(f"Sample PGD accuracy:   {pgd_correct*100:.2f}%")


Sample clean accuracy: 71.88%
Sample PGD accuracy:   29.69%


In [7]:
# -------------------------------------------------------------
# Part G: Evaluation with Deferral
# -------------------------------------------------------------

def collect_scores(loader, attack=None, attack_params=None):
    """
    Collect attention entropy scores and correctness flags for the loader.
    Returns: scores, correct_flags, labels, predictions
    """
    scores = []
    correct_flags = []
    all_labels = []
    all_preds = []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        if attack == 'fgsm':
            images = fgsm_attack(vit_model, images, labels, **attack_params)
        elif attack == 'pgd':
            images = pgd_attack(vit_model, images, labels, **attack_params)

        with torch.no_grad():
            logits, atts = vit_model(images, return_attention=True)
            entropy = compute_attention_entropy(atts)
            preds = logits.argmax(dim=1)

        scores.append(entropy.cpu())
        correct_flags.append((preds == labels).cpu())
        all_labels.append(labels.cpu())
        all_preds.append(preds.cpu())

    scores = torch.cat(scores).numpy()
    correct_flags = torch.cat(correct_flags).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_preds = torch.cat(all_preds).numpy()
    return scores, correct_flags, all_labels, all_preds

def calibrate_threshold(clean_scores, target_deferral=0.25):
    """
    Threshold on attention entropy. Higher entropy -> more uncertain -> defer.
    """
    threshold = float(np.quantile(clean_scores, 1.0 - target_deferral))
    return threshold

def calculate_metrics(scores, correct_flags, threshold):
    """
    Defer if entropy > threshold.
    """
    deferred = scores > threshold
    predicted = ~deferred
    total = len(scores)
    n_predicted = int(predicted.sum())
    n_deferred = int(deferred.sum())

    if n_predicted > 0:
        accuracy = 100.0 * correct_flags[predicted].mean()
        risk = 100.0 - accuracy
    else:
        accuracy = 0.0
        risk = 0.0

    return {
        'coverage': 100.0 * n_predicted / total,
        'deferral_rate': 100.0 * n_deferred / total,
        'accuracy_on_predicted': accuracy,
        'risk_on_predicted': risk,
        'n_predicted': n_predicted,
        'n_deferred': n_deferred,
        'total': total,
    }

# Calibrate on clean calibration split
print("\nCalibrating attention entropy threshold on clean calibration split...")
cal_clean_scores, cal_clean_correct, _, _ = collect_scores(calib_loader, attack=None)
threshold = calibrate_threshold(cal_clean_scores, target_deferral=0.25)
print(f"Calibrated threshold: {threshold:.6f}")

# Evaluate on clean evaluation split
print("\nEvaluating on clean evaluation split...")
eval_clean_scores, eval_clean_correct, _, _ = collect_scores(eval_loader, attack=None)
clean_metrics = calculate_metrics(eval_clean_scores, eval_clean_correct, threshold)
print(f"Clean metrics: {clean_metrics}")

# Evaluate on adversarial evaluation split (PGD transferred from baseline is not
# applicable here — we attack the ViT itself, since it is the object of study)
print("\nEvaluating on PGD adversarial evaluation split...")
pgd_params = {'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40}
eval_adv_scores, eval_adv_correct, _, _ = collect_scores(eval_loader, attack='pgd', attack_params=pgd_params)
adv_metrics = calculate_metrics(eval_adv_scores, eval_adv_correct, threshold)
print(f"Adversarial metrics: {adv_metrics}")



Calibrating attention entropy threshold on clean calibration split...
Calibrated threshold: 3.980342

Evaluating on clean evaluation split...
Clean metrics: {'coverage': 71.1125, 'deferral_rate': 28.8875, 'accuracy_on_predicted': np.float64(77.00826155739145), 'risk_on_predicted': np.float64(22.99173844260855), 'n_predicted': 5689, 'n_deferred': 2311, 'total': 8000}

Evaluating on PGD adversarial evaluation split...
Adversarial metrics: {'coverage': 73.0375, 'deferral_rate': 26.9625, 'accuracy_on_predicted': np.float64(20.434708197843573), 'risk_on_predicted': np.float64(79.56529180215642), 'n_predicted': 5843, 'n_deferred': 2157, 'total': 8000}


In [8]:
# -------------------------------------------------------------
# Part H: Head-to-Head Comparison with Essays #6 and #7
# -------------------------------------------------------------

reference_results = {
    'Deep Ensembles (Essay #7)':    {'type': 'Bayesian',      'adv_risk': 15.42, 'clean_risk': 10.31, 'cost': 'Medium'},
    'MC Dropout (Essay #7)':         {'type': 'Bayesian',      'adv_risk': 17.15, 'clean_risk': 12.51, 'cost': 'Low'},
    'Multi-head (Essay #6)':         {'type': 'Architecture',  'adv_risk': 19.86, 'clean_risk': None,  'cost': 'Medium'},
    'Evidential (Essay #6)':         {'type': 'Architecture',  'adv_risk': 21.31, 'clean_risk': None,  'cost': 'Medium'},
    'SWAG (Essay #7)':               {'type': 'Bayesian',      'adv_risk': 25.95, 'clean_risk': 20.94, 'cost': 'Medium'},
    'Boundary distance (Essay #6)':  {'type': 'Geometric',     'adv_risk': 77.06, 'clean_risk': None,  'cost': 'Medium'},
    'Confidence (Essay #6)':         {'type': 'Output',        'adv_risk': 78.70, 'clean_risk': None,  'cost': 'Low'},
}

print("\n" + "=" * 70)
print("Head-to-Head Comparison")
print("=" * 70)
print(f"{'Method':<35} {'Type':<15} {'Adv Risk':<12} {'Clean Risk':<12} {'Cost':<10}")
print("-" * 90)

for name, r in reference_results.items():
    cr = f"{r['clean_risk']:.2f}%" if r['clean_risk'] is not None else "—"
    print(f"{name:<35} {r['type']:<15} {r['adv_risk']:>6.2f}%      {cr:<12} {r['cost']:<10}")

print(f"{'Attention entropy (this essay)':<35} {'Attention':<15} "
      f"{adv_metrics['risk_on_predicted']:>6.2f}%      "
      f"{clean_metrics['risk_on_predicted']:>6.2f}%      {'Low':<10}")



Head-to-Head Comparison
Method                              Type            Adv Risk     Clean Risk   Cost      
------------------------------------------------------------------------------------------
Deep Ensembles (Essay #7)           Bayesian         15.42%      10.31%       Medium    
MC Dropout (Essay #7)               Bayesian         17.15%      12.51%       Low       
Multi-head (Essay #6)               Architecture     19.86%      —            Medium    
Evidential (Essay #6)               Architecture     21.31%      —            Medium    
SWAG (Essay #7)                     Bayesian         25.95%      20.94%       Medium    
Boundary distance (Essay #6)        Geometric        77.06%      —            Medium    
Confidence (Essay #6)               Output           78.70%      —            Low       
Attention entropy (this essay)      Attention        79.57%       22.99%      Low       


In [9]:
# -------------------------------------------------------------
# Part I: Summary and Outputs
# -------------------------------------------------------------

summary = {
    'clean_accuracy': clean_acc,
    'attention_entropy_mean_clean': float(np.mean(eval_clean_scores)),
    'attention_entropy_mean_adversarial': float(np.mean(eval_adv_scores)),
    'threshold': threshold,
    'clean': clean_metrics,
    'adversarial': adv_metrics,
}

with open('essay8_attention_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 55)
print("Experiment Summary")
print("=" * 55)
print(f"""
ViT Clean Accuracy: {clean_acc:.2f}%

Attention Entropy:
  - Mean on clean data:       {np.mean(eval_clean_scores):.6f}
  - Mean on adversarial data: {np.mean(eval_adv_scores):.6f}
  - Calibrated threshold:     {threshold:.6f}

Clean Metrics:
  - Coverage:  {clean_metrics['coverage']:.2f}%
  - Deferral:  {clean_metrics['deferral_rate']:.2f}%
  - Accuracy:  {clean_metrics['accuracy_on_predicted']:.2f}%
  - Risk:      {clean_metrics['risk_on_predicted']:.2f}%

Adversarial (PGD, ε=0.03) Metrics:
  - Coverage:  {adv_metrics['coverage']:.2f}%
  - Deferral:  {adv_metrics['deferral_rate']:.2f}%
  - Accuracy:  {adv_metrics['accuracy_on_predicted']:.2f}%
  - Risk:      {adv_metrics['risk_on_predicted']:.2f}%

Output files:
  - checkpoint_vit_cifar10.pth
  - essay8_attention_results.json
""")

print("\nNotebook complete.")



Experiment Summary

ViT Clean Accuracy: 76.85%

Attention Entropy:
  - Mean on clean data:       3.916901
  - Mean on adversarial data: 3.912882
  - Calibrated threshold:     3.980342

Clean Metrics:
  - Coverage:  71.11%
  - Deferral:  28.89%
  - Accuracy:  77.01%
  - Risk:      22.99%

Adversarial (PGD, ε=0.03) Metrics:
  - Coverage:  73.04%
  - Deferral:  26.96%
  - Accuracy:  20.43%
  - Risk:      79.57%

Output files:
  - checkpoint_vit_cifar10.pth
  - essay8_attention_results.json


Notebook complete.
